In [ ]:
import sys, os
import matplotlib.pyplot as plt

# Project styling
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', '..', 'src'))
from dimp.utils import init_matplotlib, get_colors
init_matplotlib()
colors = get_colors()

# Import plotting functions from the visualization script
from pann_plot import (
    print_continuous_costs,
    _build_method_solutions,
    plot_density_analysis,
)
from utils import (
    load_loss_results,
    plot_loss_results,
    plot_loss_comparison,
)

In [ ]:
data_dir = os.path.join("data", "alt_losses")
n = 160
all_results = load_loss_results(data_dir)
print(f"Loaded losses: {list(all_results.keys())}")

## Alternative Loss Functions for Time Optimization

All experiments use **ZOH3** (exact ZOH discretization + exact integrated cost via Van Loan block matrix exponential) as the base QP. Each alternative loss adds a **regularizer** $\mathcal{L}_{\text{reg}}$ to the OCP loss:

$$
\mathcal{L} = \mathcal{L}_{\text{ocp}} + \hat{\lambda} \cdot \mathcal{L}_{\text{reg}}
$$

where $\hat{\lambda}$ is adaptively balanced via EMA of gradient magnitudes (Strategy 5A).

### $\mathcal{L}_{IV}$ — Weighted Input Variation

$$
\mathcal{L}_{IV} = \sum_{k=0}^{n-2} \Delta t_k \, \| u_{k+1} - u_k \|^2
$$

Penalizes large input changes in long intervals. The $\Delta t_k$ weighting encourages the optimizer to shorten intervals where the input changes rapidly.

In [ ]:
plot_loss_results("L_IV", all_results["L_IV"], n, results_dir=None, show=True)

### $\mathcal{L}_{EQ}$ — Input Equidistribution

$$
\mathcal{L}_{EQ} = \sum_k (w_k - \bar{w})^2, \quad w_k = \| u_{k+1} - u_k \|^2
$$

Encourages uniform input variation across intervals. Unlike $\mathcal{L}_{IV}$, this does not depend on $\Delta t_k$ directly — it only asks that input changes are spread evenly.

In [ ]:
plot_loss_results("L_EQ", all_results["L_EQ"], n, results_dir=None, show=True)

### $\mathcal{L}_{CPC}$ — Constraint Proximity Concentration

$$
\mathcal{L}_{CPC} = \sum_k \sigma\!\left(\frac{|u_k| - u_{\max} + \epsilon}{\tau}\right) \Delta t_k^2
$$

Shortens intervals near active constraints. The sigmoid $\sigma$ acts as a soft indicator for constraint activity: when $|u_k|$ is close to $u_{\max}$, the penalty on $\Delta t_k^2$ drives the optimizer to place finer sampling there.

In [ ]:
plot_loss_results("L_CPC", all_results["L_CPC"], n, results_dir=None, show=True)

### $\mathcal{L}_{CSS}$ — Constraint Switching Sharpness

$$
\mathcal{L}_{CSS} = \sum_k (a_{k+1} - a_k)^2 \, \Delta t_k^2, \quad a_k = \sigma\!\left(\frac{|u_k| - u_{\max}}{\tau}\right)
$$

Concentrates samples at constraint transitions. While $\mathcal{L}_{CPC}$ targets intervals *near* constraints, $\mathcal{L}_{CSS}$ specifically targets intervals where the constraint *switches* between active and inactive.

In [ ]:
plot_loss_results("L_CSS", all_results["L_CSS"], n, results_dir=None, show=True)

### $\mathcal{L}_{\text{defect}}$ — Intra-Interval Cost Defect

$$
\mathcal{L}_{\text{defect}} = \sum_k \left( z_k^T W_k z_k - \Delta t_k (s_k^T Q s_k + u_k^T R u_k) \right)^2
$$

Penalizes the mismatch between the exact integrated cost $z_k^T W_k z_k$ (from the Van Loan computation) and the Riemann-sum approximation $\Delta t_k (s_k^T Q s_k + u_k^T R u_k)$. Large defects indicate intervals where the quadrature error is high.

In [ ]:
plot_loss_results("L_defect", all_results["L_defect"], n, results_dir=None, show=True)

### $\mathcal{L}_{\text{dyn}}$ — Dynamics Consistency

$$
\mathcal{L}_{\text{dyn}} = \sum_k \frac{\| e_k \|^2}{\Delta t_k}, \quad e_k = (A_{d,k} - I - \Delta t_k A) x_k + (B_{d,k} - \Delta t_k B) u_k
$$

Penalizes deviation of the ZOH discretization from the first-order Euler approximation. The $1/\Delta t_k$ weighting encourages shorter intervals where the discrepancy is large, i.e., where the dynamics is poorly captured by Euler.

In [ ]:
plot_loss_results("L_dyn", all_results["L_dyn"], n, results_dir=None, show=True)

### $\mathcal{L}_{\text{equi}}$ — Equidistributed Information

$$
\mathcal{L}_{\text{equi}} = \sum_k \left( \log I_k - \overline{\log I} \right)^2
$$

Log-variance of midpoint prediction error $I_k$. Encourages uniform prediction error across intervals — the idea is that if some intervals have much larger prediction errors than others, the sampling is not well distributed.

In [ ]:
plot_loss_results("L_equi", all_results["L_equi"], n, results_dir=None, show=True)

### $\mathcal{L}_{FI}$ — Fisher Information KL

$$
\mathcal{L}_{FI} = D_{KL}(q \| p), \quad q_k \propto \| \dot{x}_k \|_Q, \quad p_k = \frac{\Delta t_k}{T}
$$

Drives the timestep distribution $p$ toward the velocity-weighted target distribution $q$. The target $q$ places more weight on intervals where the state evolves quickly (high $\| \dot{x} \|_Q$), which is where finer sampling is most beneficial.

In [ ]:
plot_loss_results("L_FI", all_results["L_FI"], n, results_dir=None, show=True)

## Cross-Loss Comparison

In [ ]:
print_continuous_costs({}, all_results)

In [ ]:
plot_loss_comparison(all_results, results_dir=None, show=True)

## Sampling Density and Trajectory Change Analysis

Sampling density is defined as $(1/\Delta t_k) \cdot (T/n)$, equal to 1 for uniform sampling. Values above 1 indicate finer-than-average sampling. We compare this against trajectory change metrics ($\|\Delta u\|$, $\|\Delta s\|$) to see whether each loss successfully concentrates samples where the trajectory changes rapidly.

In [ ]:
method_solutions = _build_method_solutions({}, all_results, n)
plot_density_analysis(method_solutions, colors, results_dir=None, show=True)